# Sistema Multi-Factor de Senales de Trading con LLMs

**Curso:** Modelos de Inteligencia Artificial para Finanzas - EGADE Business School  
**Profesor:** Luis Angel Lozano Medina  
**Entrega:** 08 de junio de 2026  
**Equipo:** Mauricio Jazo, Sebastian Aceves, Jose Hernandez  

**Universo:** AAL, DAL, UAL, LUV, JBLU, ULCC | **Benchmark:** ETF JETS | **Horizonte:** 5 dias habiles

Notebook maestro que orquesta las 8 fases del pipeline end-to-end. Cada fase delega su logica a los modulos de `src/`. Ejecutar **Runtime -> Run all**; tarda ~10 minutos.

## Paso 0 - Setup y smoke test

In [ ]:
!pip install -q pandas==2.2.3 numpy==1.26.4 scipy==1.13.1 yfinance==0.2.50 finvizfinance==0.14.7 ta==0.11.0 transformers==4.46.0 torch==2.5.1 google-generativeai==0.8.3 scikit-learn==1.5.2 econml==0.15.1 joblib==1.4.2 matplotlib==3.9.2 seaborn==0.13.2

In [ ]:
import sys, os, warnings
from pathlib import Path
from datetime import datetime

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display

import data_collection, technical_indicators, custom_signals
import sentiment_analysis, signal_combiner
import causal_effects, causal_weighting
import backtesting, evaluation

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")
print("Imports OK. ROOT =", ROOT)

In [ ]:
# Carga config.py si existe, si no usa defaults
try:
    import config
    GEMINI_API_KEY = getattr(config, "GEMINI_API_KEY", "") or os.environ.get("GEMINI_API_KEY", "")
    UNIVERSE         = config.UNIVERSE
    BENCHMARK        = config.BENCHMARK
    HORIZON          = config.HORIZON
    INITIAL_CAPITAL  = config.INITIAL_CAPITAL
    DATA_START       = config.DATA_START
    BUY_THRESHOLD    = config.BUY_THRESHOLD
    SELL_THRESHOLD   = config.SELL_THRESHOLD
    CONFIDENCE_FLOOR = config.CONFIDENCE_FLOOR
    MAX_POSITION_PCT = config.MAX_POSITION_PCT
    RUN_AUTO_CLASSIFICATION = config.RUN_AUTO_CLASSIFICATION
    print("config.py cargado.")
except ImportError:
    print("config.py no encontrado - usando defaults. Crea config.py copiando config.py.example.")
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")
    UNIVERSE = ["AAL", "DAL", "UAL", "LUV", "JBLU", "ULCC"]
    BENCHMARK = "JETS"
    HORIZON, INITIAL_CAPITAL, DATA_START = 5, 100_000, "2024-06-01"
    BUY_THRESHOLD, SELL_THRESHOLD = 0.25, -0.25
    CONFIDENCE_FLOOR, MAX_POSITION_PCT = 0.60, 0.20
    RUN_AUTO_CLASSIFICATION = True

OUTPUTS_DIR = ROOT / "outputs"; OUTPUTS_DIR.mkdir(exist_ok=True)
DATA_RAW    = ROOT / "data" / "raw"
DATA_PROC   = ROOT / "data" / "processed"; DATA_PROC.mkdir(parents=True, exist_ok=True)

# Smoke test
import yfinance as yf
test = yf.download("AAL", period="5d", progress=False)
print(f"yfinance OK - {len(test)} dias de prueba bajados")
print(f"Gemini API key: {'CONFIGURADA' if GEMINI_API_KEY else 'NO CONFIGURADA (pipeline degradara a FinBERT-only)'}")

## Fase 2 - Recoleccion de noticias y precios

Carga el seed dataset (clasificado manualmente), lo augmenta con noticias auto-recolectadas de yfinance y finvizfinance, y opcionalmente auto-clasifica las augmentadas con Gemini para crecer el modeling table.

In [ ]:
# Seed: el CSV clasificado manualmente
seed_path = DATA_RAW / "airline_news_for_model.csv"
df_seed = pd.read_csv(seed_path, parse_dates=["date"])
print(f"Seed: {len(df_seed)} headlines clasificados manualmente")
print(f"  use_in_model=yes: {(df_seed['use_in_model'].str.lower() == 'yes').sum()}")

# Augmentacion
COMPANY_NAMES = {"AAL":"American Airlines","DAL":"Delta Airlines","UAL":"United Airlines",
                 "LUV":"Southwest Airlines","JBLU":"JetBlue","ULCC":"Frontier Airlines"}

print("\nAumentando con yfinance...")
df_yf = data_collection.fetch_yfinance_news(UNIVERSE, COMPANY_NAMES)
print(f"  yfinance: {len(df_yf)} headlines")

print("Aumentando con finvizfinance...")
df_fv = data_collection.fetch_finviz_news(UNIVERSE, COMPANY_NAMES)
print(f"  finviz:   {len(df_fv)} headlines")

df_aug = data_collection.consolidate_news(df_yf, df_fv)
df_aug = df_aug[~df_aug["headline"].isin(df_seed["headline"])]
print(f"\nAugmentadas (sin overlap con seed): {len(df_aug)}")

In [ ]:
# Auto-clasificacion con Gemini de las augmentadas
if RUN_AUTO_CLASSIFICATION and GEMINI_API_KEY and len(df_aug):
    df_aug_classified = sentiment_analysis.auto_classify_unlabeled(
        df_aug, api_key=GEMINI_API_KEY,
        confidence_floor=getattr(config, "AUTO_CLASS_CONFIDENCE_FLOOR", 0.5)
    )
    df_aug_classified["date"] = pd.to_datetime(df_aug_classified["date"])
    df_all = pd.concat([df_seed, df_aug_classified], ignore_index=True)
    df_all = df_all.drop_duplicates(subset=["ticker","headline"])
else:
    print("Saltando auto-clasificacion (sin API key o RUN_AUTO_CLASSIFICATION=False).")
    df_all = df_seed.copy()

# Filtra al modelo
df_model = df_all[df_all["use_in_model"].astype(str).str.lower().str.strip() == "yes"].copy().reset_index(drop=True)
df_model["date"] = pd.to_datetime(df_model["date"])
print(f"\nTotal headlines para el modelo: {len(df_model)}")
print(df_model["ticker"].value_counts())

## Fase 4 - Sentiment con FinBERT + Gemini

FinBERT (ProsusAI) corre sobre todos los headlines. Si hay API key de Gemini, tambien se re-clasifica con prompt structured-output (event_type + sentiment_score + confidence). El sentiment final es hibrido: Gemini si su confianza >= 0.5, FinBERT en otro caso.

In [ ]:
# FinBERT
print("Corriendo FinBERT (primera vez tarda ~30s)...")
fb = sentiment_analysis.score_finbert(df_model["headline"].tolist())
df_model = pd.concat([df_model.reset_index(drop=True), fb], axis=1)
print(f"FinBERT OK - {len(df_model)} headlines")

# Gemini (opcional)
if GEMINI_API_KEY:
    llm = sentiment_analysis.score_gemini(df_model, api_key=GEMINI_API_KEY)
    df_model = pd.concat([df_model.reset_index(drop=True), llm], axis=1)
    print(f"Gemini OK")
else:
    print("Sin Gemini API key - solo FinBERT.")

# Hibrido
df_model = sentiment_analysis.hybrid_sentiment(df_model, llm_confidence_threshold=0.5)
print("\nFuente del sentiment final:")
print(df_model["sentiment_source"].value_counts())

In [ ]:
# Comparacion FinBERT vs Gemini
comp = sentiment_analysis.compare_finbert_vs_gemini(df_model)
if comp.get("available"):
    print(f"Pearson  FinBERT vs Gemini : {comp['pearson']:+.3f}")
    print(f"Spearman FinBERT vs Gemini : {comp['spearman']:+.3f}")
    print(f"Coincidencia de signo      : {comp['sign_agreement_pct']:.1%}  (n={comp['n']})")

    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].scatter(df_model["finbert_sentiment"], df_model["llm_sentiment_score"], alpha=0.7, s=40, edgecolor="white")
    ax[0].plot([-1,1], [-1,1], "--", color="lightgray")
    ax[0].axhline(0, color="grey", lw=0.6); ax[0].axvline(0, color="grey", lw=0.6)
    ax[0].set_xlabel("FinBERT"); ax[0].set_ylabel("Gemini")
    ax[0].set_title(f"FinBERT vs Gemini (Spearman {comp['spearman']:+.3f})")

    ax[1].hist([df_model["finbert_sentiment"], df_model["llm_sentiment_score"]], bins=12,
               label=["FinBERT","Gemini"], alpha=0.7)
    ax[1].set_xlabel("Sentiment score"); ax[1].set_title("Distribuciones"); ax[1].legend()
    plt.tight_layout(); plt.savefig(OUTPUTS_DIR / "llm_vs_finbert.png", dpi=120, bbox_inches="tight")
    plt.show()
else:
    print("Gemini no corrio - sin comparacion.")

## Fase 5a - Anti-leakage timestamp alignment

Asumimos noticias after-close → siguiente dia habil. Nunca introduce look-ahead.

In [ ]:
def shift_to_next_business_day(d):
    return (pd.Timestamp(d) + pd.tseries.offsets.BDay(1)).normalize()

df_model["effective_date"] = df_model["date"].apply(shift_to_next_business_day)
print(df_model[["date","effective_date","ticker"]].head(4))

## Fase 5b - Agregacion a ticker-fecha

Una fila por (ticker, effective_date) con sentiment medio, confidence-weighted, recency-weighted, sector_sentiment, relative_sentiment y count.

In [ ]:
HALFLIFE = 3

def aggregate_ticker_date(g):
    n = len(g)
    s = g["sentiment_score"].values
    conf = g["confidence"].values
    days_old = (g["effective_date"] - g["date"]).dt.days.values
    w = np.exp(-days_old / HALFLIFE)
    return pd.Series({
        "news_count":      n,
        "mean_sentiment":  s.mean() if n else 0,
        "positive_share":  (s > 0).mean() if n else 0,
        "negative_share":  (s < 0).mean() if n else 0,
        "confidence_weighted_sentiment": (s*conf).sum() / (conf.sum() if conf.sum() else 1),
        "recency_weighted_sentiment":    (s*w).sum() / (w.sum() if w.sum() else 1),
        "main_event_type": (g["event_type_final"].astype(str).mode().iat[0] if n else "unknown"),
    })

# Individuales vs sectoriales
mask_ind = df_model["relevance_level"].astype(str).str.lower().isin(["direct", "indirect"])
df_ind = df_model[mask_ind]
df_sec = df_model[df_model["relevance_level"].astype(str).str.lower() == "sector"]

with warnings.catch_warnings():
    warnings.simplefilter("ignore", FutureWarning)
    signals_news = df_ind.groupby(["ticker", "effective_date"], as_index=False).apply(aggregate_ticker_date).reset_index(drop=True)

# Sector sentiment por fecha
df_sec["effective_date"] = df_sec["date"].apply(shift_to_next_business_day)
sector = (df_sec.drop_duplicates(subset=["effective_date","headline"])
          .groupby("effective_date", as_index=False)
          .agg(sector_sentiment=("sentiment_score","mean"),
               sector_news_count=("sentiment_score","size")))

signals_news = signals_news.rename(columns={"effective_date":"date"})
sector = sector.rename(columns={"effective_date":"date"})
signals_news = signals_news.merge(sector, on="date", how="left")
signals_news["sector_sentiment"] = signals_news["sector_sentiment"].fillna(0)
signals_news["sector_news_count"] = signals_news["sector_news_count"].fillna(0).astype(int)
signals_news["relative_sentiment"] = signals_news["mean_sentiment"] - signals_news["sector_sentiment"]
signals_news["sentiment_signal"] = signals_news["mean_sentiment"]   # alias para signal_combiner
print(f"Signals ticker-fecha (noticias): {len(signals_news)}")
signals_news.head()

## Fase 2/3 - Precios, indicadores tecnicos y senales propias

In [ ]:
# Precios
prices = data_collection.fetch_prices(UNIVERSE + [BENCHMARK], start=DATA_START)
print(f"Precios cargados: {list(prices.keys())}")

# Controles de precio (past returns, vol, volume)
controls = data_collection.compute_price_controls(prices, BENCHMARK)
print(f"Controles: {controls.shape}")
controls.head()

In [ ]:
# Indicadores tecnicos por ticker
techs = []
for tk in UNIVERSE:
    df_t = prices[tk]
    ind  = technical_indicators.compute_all_indicators(df_t)
    agg  = technical_indicators.aggregate_by_category(ind)
    agg["ticker"] = tk
    agg["date"]   = agg.index
    techs.append(agg.reset_index(drop=True))
df_tech = pd.concat(techs, ignore_index=True)
print(f"Tecnicos: {df_tech.shape}")
df_tech.head()

In [ ]:
# Senales propias
custs = []
bench_close = prices[BENCHMARK]["Close"]
for tk in UNIVERSE:
    cs = custom_signals.compute_custom_signals(prices[tk]["Close"], bench_close)
    cs["custom_signal"] = custom_signals.aggregate_custom_signal(cs)
    cs["ticker"] = tk; cs["date"] = cs.index
    custs.append(cs.reset_index(drop=True))
df_cust = pd.concat(custs, ignore_index=True)
print(f"Custom signals: {df_cust.shape}")
df_cust[["ticker","date","mr_zscore","mtf_momentum","rel_strength","custom_signal"]].head()

## Fase 5 - Modeling table

Merge de noticias + tecnicos + customs + controles + target. El target es `future_excess_return_5d = stock_return_5d - JETS_return_5d`.

In [ ]:
# Merge tecnicos + customs por ticker-fecha
df_features = df_tech.merge(
    df_cust[["ticker","date","custom_signal"]],
    on=["ticker","date"], how="outer"
)
df_features["date"] = pd.to_datetime(df_features["date"])
df_features = df_features.merge(controls, on=["ticker","date"], how="left")

# Merge con sentiment
df_features = df_features.merge(signals_news, on=["ticker","date"], how="left")
df_features["sentiment_signal"] = df_features["sentiment_signal"].fillna(0)

# Build target: future excess return a 5 dias habiles
px = pd.concat({tk: prices[tk]["Close"] for tk in UNIVERSE + [BENCHMARK]}, axis=1)
px.columns = px.columns.get_level_values(0)
fwd = px.shift(-HORIZON) / px - 1

def future_excess(row):
    if row["ticker"] not in fwd.columns or row["date"] not in fwd.index:
        return np.nan
    s = fwd.loc[row["date"], row["ticker"]]
    b = fwd.loc[row["date"], BENCHMARK]
    return s - b

df_features["future_excess_return_5d"] = df_features.apply(future_excess, axis=1)
df_features["next_return"] = df_features.groupby("ticker")["past_5d_return"].shift(-1) / 5  # approx 1d
df_features["outperform_label"] = np.where(
    df_features["future_excess_return_5d"].isna(), np.nan,
    (df_features["future_excess_return_5d"] > 0).astype(float)
)

# Modeling table final
modeling_table = df_features.dropna(subset=["future_excess_return_5d"]).copy().reset_index(drop=True)
modeling_table.to_csv(DATA_PROC / "modeling_table.csv", index=False)
print(f"Modeling table: {modeling_table.shape}")
print(f"Tickers: {sorted(modeling_table.ticker.unique())}")
print(f"Date range: {modeling_table.date.min().date()} -> {modeling_table.date.max().date()}")

## Fase 5 - Signal combiner: base + 3 experimentos de pesos

In [ ]:
experiments = signal_combiner.run_weight_experiments(modeling_table,
    buy_threshold=BUY_THRESHOLD, sell_threshold=SELL_THRESHOLD)

print("Distribucion de senales por configuracion:")
for name, df_e in experiments.items():
    counts = df_e["signal_discrete"].value_counts()
    print(f"  {name:18s}  BUY {counts.get('BUY',0):>4d}  HOLD {counts.get('HOLD',0):>4d}  SELL {counts.get('SELL',0):>4d}")

# Usamos el 'base' como modelo principal
modeling_table = experiments["base"]

## Fase 5.5 - Causal HTE

Estimamos el efecto causal de cada familia de senales sobre el next_return, condicionado al regimen de mercado (ATR%, ADX, volumen, etc.). Luego ajustamos los pesos dinamicamente.

In [ ]:
SIGNAL_COLS = ["sentiment_signal","momentum_signal","trend_signal",
               "volatility_signal","volume_signal","custom_signal"]
CONTEXT_COLS = ["atr_pct","adx_signal","volume_spike",
                "past_5d_return","past_20d_return","volatility_20d",
                "volume_change_20d","market_return_20d","news_count"]

# Asegurar que todas las columnas de contexto existan
for c in CONTEXT_COLS:
    if c not in modeling_table.columns:
        modeling_table[c] = 0.0

# next_return como outcome (fallback: future_excess_return_5d / 5 si next_return tiene NaN)
modeling_table["next_return"] = modeling_table["next_return"].fillna(
    modeling_table["future_excess_return_5d"] / 5)

cfg = causal_effects.CausalConfig(outcome_col="next_return", min_samples=20, min_samples_leaf=3)
est = causal_effects.CausalEffectsEstimator(
    signal_cols=SIGNAL_COLS, context_cols=CONTEXT_COLS, config=cfg)

try:
    est.fit(modeling_table)
    print("Causal model entrenado")
    ate_table = est.ate_summary()
    print("\nATE por senal:")
    print(ate_table.round(5))
    ate_table.to_csv(OUTPUTS_DIR / "hte_segments_table.csv", index=False)
except Exception as e:
    print(f"Causal fit fallo: {e}")
    ate_table = pd.DataFrame()
    est = None

In [ ]:
# Predict effects + causal weights + dynamic combine
if est is not None:
    modeling_table = est.predict_effects(modeling_table)
    base_w = signal_combiner.DEFAULT_WEIGHTS
    modeling_table = causal_weighting.compute_causal_weights(
        modeling_table, SIGNAL_COLS, base_w, gamma=1.0)
    modeling_table = causal_weighting.combine_signals_with_causal_weights(
        modeling_table, SIGNAL_COLS,
        buy_threshold=BUY_THRESHOLD, sell_threshold=SELL_THRESHOLD)

    # Heatmap de pesos causales por senal (promedio en el periodo)
    weight_cols = [f"causal_weight_{s}" for s in SIGNAL_COLS]
    by_signal = modeling_table[weight_cols].mean().rename(lambda c: c.replace("causal_weight_",""))
    base_arr  = pd.Series(base_w)[by_signal.index]
    cmp_df = pd.DataFrame({"base": base_arr, "causal_mean": by_signal,
                           "delta": by_signal - base_arr}).round(4)
    print(cmp_df)

    # Heatmap de causal weights por senal en el tiempo (resampled mensual)
    mt = modeling_table.copy()
    mt["month"] = pd.to_datetime(mt["date"]).dt.to_period("M")
    hm = (mt.groupby("month")[weight_cols].mean()
          .rename(columns=lambda c: c.replace("causal_weight_","")))
    fig, ax = plt.subplots(figsize=(10, 4.5))
    sns.heatmap(hm.T, cmap="RdBu_r", center=0, cbar_kws={"label":"Peso causal"}, ax=ax)
    ax.set_xlabel("Mes"); ax.set_ylabel("Senal")
    ax.set_title("Pesos causales dinamicos por mes")
    plt.tight_layout(); plt.savefig(OUTPUTS_DIR / "causal_weight_heatmap.png", dpi=120)
    plt.show()
else:
    print("Sin modelo causal - skip dynamic weighting.")

## Fase 6 - Backtesting

Tres estrategias: (A) base con pesos fijos, (B) sentiment_heavy, (C) pesos dinamicos causal. Todas vs buy & hold del JETS.

In [ ]:
# Necesitamos next_return para el backtest (lo definimos por accion al dia)
# Si esta vacio, usamos future_excess_return_5d / 5 como proxy diario
modeling_table["next_return"] = modeling_table["next_return"].fillna(
    modeling_table["future_excess_return_5d"] / 5)

# Backtests
bt_base = backtesting.backtest_strategy(
    modeling_table.rename(columns={"signal_discrete":"signal_discrete"}),
    signal_col="signal_discrete", initial_capital=INITIAL_CAPITAL)

bt_sentiment_heavy = backtesting.backtest_strategy(
    experiments["sentiment_heavy"], signal_col="signal_discrete",
    initial_capital=INITIAL_CAPITAL)

if "signal_discrete_causal" in modeling_table.columns:
    bt_causal = backtesting.backtest_strategy(
        modeling_table, signal_col="signal_discrete_causal",
        initial_capital=INITIAL_CAPITAL)
else:
    bt_causal = bt_base.copy()

# Buy & hold del JETS
bh = backtesting.backtest_buy_and_hold(prices[BENCHMARK]["Close"],
                                       initial_capital=INITIAL_CAPITAL)

# Equity curves
backtesting.plot_equity_curve(bt_base, bh,
    title="Equity Curve - Modelo Base vs Buy & Hold",
    out_path=str(OUTPUTS_DIR / "equity_curve.png"))

backtesting.plot_equity_curve(bt_causal, bh,
    title="Equity Curve - Modelo Causal vs Buy & Hold",
    out_path=str(OUTPUTS_DIR / "causal_vs_base_backtest.png"))

## Fase 7 - Metricas de clasificacion

In [ ]:
# Ground truth: next_return con umbral de 1%
gt = evaluation.make_ground_truth(modeling_table["next_return"], threshold_pct=1.0)
pred = modeling_table["signal_discrete"]

# Confusion matrix
evaluation.plot_confusion_matrix(gt, pred, out_path=str(OUTPUTS_DIR / "confusion_matrix.png"))

# Metricas
metrics_cls = evaluation.classification_metrics(gt, pred)
print("Metricas de clasificacion:")
for k, v in metrics_cls.items():
    print(f"  {k:20s}: {v:.3f}")

In [ ]:
# ROC y KS - binarizamos en movimiento positivo
y_bin = evaluation.binarize_movement(modeling_table["next_return"])
auc = evaluation.plot_roc(y_bin, modeling_table["signal_raw"],
                          out_path=str(OUTPUTS_DIR / "roc_curve.png"))
ks  = evaluation.ks_statistic(y_bin, modeling_table["signal_raw"])
print(f"\nROC-AUC : {auc:.3f}")
print(f"KS      : {ks:.3f}")

## Fase 8 - Gestion de riesgo

Reliability diagram (calibracion), position sizing y stop-loss.

In [ ]:
# Calibracion
hit = (modeling_table["signal_discrete"] == gt).astype(int)
cal = evaluation.plot_calibration(modeling_table["confidence"], hit,
    out_path=str(OUTPUTS_DIR / "calibration_plot.png"))
print(cal.round(3))

In [ ]:
# Position sizing
modeling_table["pos_size"] = evaluation.position_size(
    modeling_table["confidence"], modeling_table["volatility_20d"],
    max_position_pct=MAX_POSITION_PCT)

# Confidence filter
filtered = evaluation.confidence_filter(modeling_table, threshold=CONFIDENCE_FLOOR)
print(f"Senales antes del filtro: {(modeling_table['signal_discrete'] != 'HOLD').sum()}")
print(f"Senales tras filtro de confianza ({CONFIDENCE_FLOOR}): {(filtered['signal_filtered'] != 'HOLD').sum()}")

# Stops
stopped = evaluation.stop_loss_rules(modeling_table)
print(f"Trades cerrados por stop_flip:   {stopped['stop_flip'].sum()}")
print(f"Trades cerrados por low_conf:    {stopped['stop_low_conf'].sum()}")

## Comparacion de modelos A vs B vs C

Modelo A: pesos base de la rubrica.  
Modelo B: pesos sentiment_heavy (experimento).  
Modelo C: pesos dinamicos con HTE causal.

In [ ]:
comp = evaluation.compare_models(
    {"A. Base":           bt_base,
     "B. Sentiment Heavy": bt_sentiment_heavy,
     "C. Causal HTE":     bt_causal},
    benchmark_bt=bh, initial_capital=INITIAL_CAPITAL)
print(comp.round(4))
comp.to_csv(OUTPUTS_DIR / "model_comparison.csv")

## Signal Card final

Se completa automaticamente con las metricas calculadas.

In [ ]:
# Estadisticos clave
n_signals = len(modeling_table)
n_obs_eval = modeling_table["future_excess_return_5d"].notna().sum()
sent_source = "Gemini + FinBERT hibrido" if GEMINI_API_KEY else "FinBERT solo"

filas = [
    ("Curso",            "Modelos de Inteligencia Artificial para Finanzas - EGADE"),
    ("Profesor",         "Luis Angel Lozano Medina"),
    ("Equipo",           "Mauricio Jazo, Sebastian Aceves, Jose Hernandez"),
    ("Entrega",          "08 de junio de 2026"),
    ("Universo",         ", ".join(UNIVERSE)),
    ("Benchmark",        f"{BENCHMARK} (ETF sectorial)"),
    ("Horizonte",        f"{HORIZON} dias habiles"),
    ("Sentiment engine", sent_source),
    ("Headlines en modelo",  str(len(df_model))),
    ("Obs ticker-fecha total", str(n_signals)),
    ("Obs evaluables (5d cumplidos)", str(n_obs_eval)),
    ("Modelo A - Total Return",  f"{comp.loc['A. Base','Total Return']:+.2%}"),
    ("Modelo A - Sharpe",        f"{comp.loc['A. Base','Sharpe']:+.2f}"),
    ("Modelo A - Max Drawdown",  f"{comp.loc['A. Base','Max Drawdown']:+.2%}"),
    ("Modelo C - Total Return",  f"{comp.loc['C. Causal HTE','Total Return']:+.2%}"),
    ("Modelo C - Sharpe",        f"{comp.loc['C. Causal HTE','Sharpe']:+.2f}"),
    ("Buy & Hold JETS - Return", f"{comp.loc['A. Base','Benchmark Return']:+.2%}"),
    ("ROC-AUC (signal_raw vs movimiento positivo)", f"{auc:.3f}"),
    ("KS statistic",                                f"{ks:.3f}"),
    ("Accuracy clasificacion",                      f"{metrics_cls['Accuracy']:.1%}"),
    ("F1 Macro",                                    f"{metrics_cls['F1 Macro']:.3f}"),
]

nl = chr(10)
tabla = "### Signal Card - Sistema Multi-Factor Aerolineas" + nl*2
tabla += "| Campo | Detalle |" + nl + "|---|---|" + nl
tabla += nl.join("| **" + k + "** | " + v + " |" for k, v in filas)

notas = [
    "**Limitaciones a documentar:**",
    "",
    "- Sin costos de transaccion, slippage ni impacto de mercado en el backtest.",
    "- Causal inference observacional: confounders relevantes asumidos en X, no prueba causal definitiva.",
    "- Muestra de noticias frozen a la fecha de recoleccion; no es reproducible end-to-end desde RSS.",
    "- Sector sentiment separado evita doble conteo de articulos sectoriales.",
    "- Position sizing y stops definidos como reglas; no se backtested la version completa con sizing dinamico.",
]
display(Markdown(tabla + nl*2 + nl.join(notas)))

# Tambien lo guardamos en outputs/
(OUTPUTS_DIR / "signal_card.md").write_text(tabla + nl*2 + nl.join(notas), encoding="utf-8")
print("\nSignal card guardado en outputs/signal_card.md")

## Outputs guardados

In [ ]:
print("Archivos generados en outputs/:")
for f in sorted(OUTPUTS_DIR.glob("*")):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:40s}  {size_kb:>7.1f} KB")
print("\nArchivos generados en data/processed/:")
for f in sorted(DATA_PROC.glob("*")):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:40s}  {size_kb:>7.1f} KB")
print("\nPipeline completado.")